### Copyright (C) Infineon Technologies AG 2025
 
Copyright (c) 2025, Infineon Technologies AG, or an affiliate of Infineon Technologies AG. All rights reserved.
This software, associated documentation and materials ("Software") is owned by Infineon Technologies AG or one of its affiliates ("Infineon") and is protected by and subject to worldwide patent protection, worldwide copyright laws, and international treaty provisions. Therefore, you may use this Software only as provided in the license agreement accompanying the software package from which you obtained this Software. If no license agreement applies, then any use, reproduction, modification, translation, or compilation of this Software is prohibited without the express written permission of Infineon.

Disclaimer: UNLESS OTHERWISE EXPRESSLY AGREED WITH INFINEON, THIS SOFTWARE IS PROVIDED AS-IS, WITH NO WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING, BUT NOT LIMITED TO, ALL WARRANTIES OF NON-INFRINGEMENT OF THIRD-PARTY RIGHTS AND IMPLIED WARRANTIES SUCH AS WARRANTIES OF FITNESS FOR A SPECIFIC USE/PURPOSE OR MERCHANTABILITY. Infineon reserves the right to make changes to the Software without notice. You are responsible for properly designing, programming, and testing the functionality and safety of your intended application of the Software, as well as complying with any legal requirements related to its use. Infineon does not guarantee that the Software will be free from intrusion, data theft or loss, or other breaches ("Security Breaches"), and Infineon shall have no liability arising out of any Security Breaches. Unless otherwise explicitly approved by Infineon, the Software may not be used in any application where a failure of the Product or any consequences of the use thereof can reasonably be expected to result in personal injury.

# New Model Template

This template is for adding a pretrained model that is already ready for inference and deployment. It demonstrates how to define the model, provide representative input data, validate the exported ONNX model, convert it to AURIX™ deployment artifacts, and inspect target performance. It intentionally does not train a model; replace the marked examples with your model and input data.

## Notebook Structure

1. **Imports and Helper Functions**
2. **Pretrained Model Definition**
3. **Input Data and Inference Validation**
4. **Export and Save the Model and Data**
5. **Convert the Model for AURIX™ Deployment**
6. **Display Inference Timing and Platform Benchmark Results**
7. **References and Generated Artifacts**

In [ ]:
# TO DO: remove libs you don't need and add libs you need

import sys
import os

# Add the parent directory to the Python path so we can import from _CentralScripts
parent_dir = os.path.dirname(os.getcwd())
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

import _CentralScripts.helper_functions as cs

## Pretrained Model Definition

Replace the example model with your pretrained model and set its framework origin. This section defines the model for inference; model training is intentionally outside the scope of this template. The conversion target demonstrated below is AURIX™.

In [ ]:
from modelling_helper import get_model

# TO DO: add your model in modelling helper
origin = "tf"  # choose your origin, e.g., "tf" for TensorFlow, "torch" for PyTorch

# TO DO: choose your model_name
model_name = "new_model"

# TO DO: add your model
model = get_model(origin)

## Input Data and Inference Validation

Provide a representative input for the pretrained model as a NumPy array. This is user-provided example input, not a fixed training dataset. It is used to generate a reference output and validate that the exported ONNX model produces the expected result.

Document the input shape, dtype, value range, preprocessing, and expected output format when replacing this example.

In [ ]:
import numpy as np

# TO DO: add an exemplary input as numpy array
input_target = np.random.rand(10).astype(np.float32)  # Example input for the model
output_target = cs.get_predictions(origin, model, input_target)

## Export and Save the Model and Data

Export the pretrained model to ONNX and save the representative input and reference output as `.pb` files. These artifacts are used by the conversion service to validate generated deployment code.

In [ ]:
# export model and data as ONNX and .pb
cs.save_all(model_name, input_target, output_target, model, origin)
cs.test_onnx_pb(model_name)

## Convert the Model for AURIX™ Deployment

- Make sure the Docker container with conversion tools is running.
- Submit the ONNX model together with the representative input and output data.
- Results are saved in the `out/<model_name>/test_<model_name>/<target>/` folder and include generated code, binaries, logs, and timing data.

In [ ]:
cs.ensure_docker_container()

In [ ]:
from _CentralScripts.python_flask_client import CallTools

model_folder, onnx_model_file = cs.get_output_paths(model_name)

target = "tc4dx"  # Available targets: "tc3xx", "tc4dx", "tc4dx_ppu", "arm_m7", "arm_m4"

tool = CallTools(
    folder=model_folder,
    url="http://localhost:8080/convert",
    target=target,
    profile=True,
)
tool.convert_model()

## Display Inference Timing and Platform Benchmark Results

The timing output summarizes estimated cycles, instructions, average CPI, latency, and throughput for the selected target. The platform comparison displays conversion and execution results across the supported hardware targets.

In [ ]:
cs.display_inference_benchmark(model_folder, target)

In [ ]:
cs.plot_profile_shares(model_folder, target)

### Platform Comparison

Compare the generated model's conversion and execution performance across the supported hardware platforms.

In [ ]:
target = "compare"

tool = CallTools(
    folder=model_folder,
    url="http://localhost:8080/convert",
    target=target,
)
tool.convert_model()

In [ ]:
cs.display_platform_comparison(model_folder)

## References and Generated Artifacts

- **Input source:** User-provided representative input data; this template does not use a fixed training dataset.
- **Model:** A pretrained model supplied through `modelling_helper.py`.
- **Validation artifacts:** ONNX model, representative input `.pb`, and reference output `.pb`.
- **Deployment artifacts:** Generated C code, binaries, logs, `results.json`, inference timing, and platform comparison results, saved under `out/new_model/` (replace `new_model` with your chosen `model_name`).
- **Deployment targets:** AURIX TC3x, AURIX TC4x, AURIX TC4x PPU, and TRAVEO Cortex M4, selectable in the conversion configuration.